In [1]:
from config import settings
import json
from datetime import datetime, timezone, timedelta
import requests
from bank_client import jwt_gen

import os
os.chdir("..")

In [5]:
API_ORIGIN = "https://api.enablebanking.com"
ASPSP_NAME = "Revolut"
ASPSP_COUNTRY = "HU"

with open(settings.sessions_info_path, "r") as f:
    session_info = json.load(f)

base_headers = jwt_gen()

# Using the first available account for the following API calls
account_uid = session_info["Revolut"]['accounts'][0]['uid']

all_transactions = []

# Retrieving account balances
r = requests.get(f"{API_ORIGIN}/accounts/{account_uid}/balances", headers=base_headers)
if r.status_code == 200:
    print("Balances:")
    print(r.json())
else:
    print(f"Error response {r.status_code}:", r.text)


# Retrieving account transactions (since 90 days ago)
query = {
    "date_from": (datetime.now(timezone.utc) - timedelta(days=90)).date().isoformat(),
}
continuation_key = None
while True:
    if continuation_key:
        query["continuation_key"] = continuation_key
    r = requests.get(
        f"{API_ORIGIN}/accounts/{account_uid}/transactions",
        params=query,
        headers=base_headers,
    )
    if r.status_code == 200:
        resp_data = r.json()
        all_transactions.extend(resp_data["transactions"])
        continuation_key = resp_data.get("continuation_key")
        if not continuation_key:
            print("No continuation key. All transactions were fetched")
            break
        print(f"Going to fetch more transactions with continuation key {continuation_key}")
    else:
        print(f"Error response {r.status_code}:", r.text)

with open("./data/transactions_revolut.json", "w") as f:
    json.dump(all_transactions, f, indent=2)

print("All done!")

Balances:
{'balances': [{'name': 'Available balance calculated in the course of the business day', 'balance_amount': {'currency': 'HUF', 'amount': '115959.85'}, 'balance_type': 'ITAV', 'last_change_date_time': None, 'reference_date': '2026-06-18', 'last_committed_transaction': None}]}
Going to fetch more transactions with continuation key eyJwYXJhbXMiOnsiYWNjb3VudElkIjoiM2M5MzVhNWMtN2ZkOS00NTc4LThhZTAtNDJkMGMxM2M1NjBjIiwiZGF0ZUZyb20iOiIyMDI2LTAzLTIwIiwiZGF0ZVRvIjpudWxsLCJ0cmFuc2FjdGlvblN0YXR1cyI6bnVsbH0sInBhdGgiOiJodHRwczovL29iYS1hdXRoLnJldm9sdXQuY29tL2FjY291bnRzLzNjOTM1YTVjLTdmZDktNDU3OC04YWUwLTQyZDBjMTNjNTYwYy90cmFuc2FjdGlvbnM/ZnJvbUJvb2tpbmdEYXRlVGltZT0yMDI2LTAzLTIwVDExOjA5OjA0LjUwNCZ0b0Jvb2tpbmdEYXRlVGltZT0yMDI2LTA2LTA3VDE0OjQyOjQ2LjMzODA2MCJ9.e4670fd16148801e7f0929fa4101f13f509bce680c9bade1e4228abd088714f8
Going to fetch more transactions with continuation key eyJwYXJhbXMiOnsiYWNjb3VudElkIjoiM2M5MzVhNWMtN2ZkOS00NTc4LThhZTAtNDJkMGMxM2M1NjBjIiwiZGF0ZUZyb20iOiIyMDI2LTAzLTIwIiwiZGF0ZV

In [5]:
settings.BANKS[0]["name"]

'Erste Bank'